# Ford VinGuard / Dock 360 - Pipeline completo

Notebook consolidado da US-011 para execucao ponta a ponta: carrega dados, EDA, clustering, classificacao de perfis, churn, MLflow e conclusoes para a banca.


## 1. Carrega dados

A base historica contem o resultado completo de relacionamento e manutencao; a base operacional contem somente atributos disponiveis no momento da compra. Para evitar data leakage, os modelos supervisionados usam a base operacional ou removem explicitamente as colunas de comportamento posterior.


In [ ]:
import matplotlib
matplotlib.use('Agg')

import os
from importlib.machinery import SourceFileLoader
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import adjusted_rand_score

os.makedirs('reports', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

HISTORICO_PATH = Path('data/raw/ford_clientes_historico_completo.csv')
OPERACIONAL_PATH = Path('data/raw/ford_clientes_operacional_compra.csv')
CLUSTER_LABELS_PATH = Path('data/processed/cluster_labels')

historico = pd.read_csv(HISTORICO_PATH)
operacional = pd.read_csv(OPERACIONAL_PATH)
print(f'Base historica: {historico.shape[0]:,} linhas x {historico.shape[1]} colunas')
print(f'Base operacional: {operacional.shape[0]:,} linhas x {operacional.shape[1]} colunas')


## 2. EDA

A analise exploratoria resume qualidade, distribuicao dos quatro perfis e comportamento de manutencao. As visualizacoes sao salvas em `reports/` para uso direto no relatorio e na apresentacao.


In [ ]:
perfil_counts = historico['perfil_latente'].value_counts().rename_axis('perfil').reset_index(name='clientes')
perfil_counts['pct'] = perfil_counts['clientes'] / len(historico)
display(perfil_counts)

eda_cols = [
    'renda_mensal', 'score_credito', 'preco_veiculo', 'km_estimado_ano',
    'share_revisoes_rede_24m', 'qtde_revisoes_24m', 'gasto_manutencao_rede_24m',
    'satisfacao_marca_24m', 'churn_rede_24m'
]
display(historico[eda_cols].describe().T.round(2))

plt.figure(figsize=(8, 4))
perfil_counts.plot.bar(x='perfil', y='clientes', legend=False, color='#003478')
plt.title('Distribuicao dos perfis latentes')
plt.xlabel('Perfil')
plt.ylabel('Clientes')
plt.tight_layout()
plt.savefig('reports/pipeline_distribuicao_perfis.png', dpi=150, bbox_inches='tight')
plt.close()

display(Image(filename='reports/pipeline_distribuicao_perfis.png'))


## 3. Clustering

O K-Means usa apenas variaveis de pos-venda para segmentar comportamento real de manutencao. O pre-processamento fica dentro de `sklearn.Pipeline`, com imputacao, escala e K-Means juntos para evitar transformacoes divergentes entre treino e inferencia.


In [ ]:
clustering = SourceFileLoader('clustering', 'src/pipeline/clustering.py').load_module()

required_cluster_outputs = [
    CLUSTER_LABELS_PATH,
    Path('reports/elbow_silhouette'),
    Path('reports/clusters_pca'),
]
if not all(path.exists() for path in required_cluster_outputs):
    labels_df, _ = clustering.run_clustering(str(HISTORICO_PATH))
else:
    labels_df = pd.read_csv(CLUSTER_LABELS_PATH)

clusters = historico[['id_cliente', 'perfil_latente']].merge(
    labels_df,
    left_on='id_cliente',
    right_on='cliente_id',
    how='inner',
)
ari = adjusted_rand_score(clusters['perfil_latente'], clusters['perfil_cluster'])
print(f'ARI vs perfil_latente: {ari:.4f}')
display(pd.crosstab(clusters['perfil_cluster'], clusters['perfil_latente'], normalize='index').round(3))

for image_path in ['reports/elbow_silhouette', 'reports/clusters_pca']:
    if Path(image_path).exists():
        display(Image(filename=image_path))


## 4. Classificacao de perfis

A classificacao de perfil usa atributos de compra e cadastro, removendo todas as colunas proibidas de leakage. A metrica principal e F1 Macro porque os perfis nao tem a mesma frequencia e a avaliacao precisa tratar as quatro classes com peso equivalente.


In [ ]:
train_classifier = SourceFileLoader('train_classifier', 'src/pipeline/train_classifier.py').load_module()
visualizations = SourceFileLoader('visualizations', 'src/pipeline/visualizations.py').load_module()

comparison_path = Path('reports/model_comparison')
if not comparison_path.exists():
    comparison = train_classifier.train_all_models(str(HISTORICO_PATH))
else:
    comparison = pd.read_csv(comparison_path)

if not Path('reports/feature_importance.csv').exists() or not Path('reports/confusion_matrix_rf.png').exists():
    visualizations.main(str(HISTORICO_PATH))

best_f1_macro = comparison['f1_macro_mean'].max()
best_model = comparison.sort_values('f1_macro_mean', ascending=False).iloc[0]['model']
print(f'Melhor classificador: {best_model} | F1 Macro CV: {best_f1_macro:.4f}')
display(comparison)
display(pd.read_csv('reports/feature_importance.csv').head(10))

for image_path in ['reports/feature_importance.png', 'reports/confusion_matrix_rf.png']:
    if Path(image_path).exists():
        display(Image(filename=image_path))


## 5. Classificacao de churn

O churn e treinado a partir da base operacional de compra, juntando apenas o alvo `churn_rede_24m` da base historica. AUC-ROC e usada para medir a capacidade de ranquear risco antes da acao de retencao no Dock 360.


In [ ]:
train_churn = SourceFileLoader('train_churn', 'src/pipeline/train_churn.py').load_module()

metric_files = list(Path('mlruns').glob('**/metrics/auc_roc_test')) if Path('mlruns').exists() else []
if not Path('models/churn_rf_calibrated.joblib').exists() or not Path('reports/precision_recall_churn').exists():
    _, auc_roc = train_churn.train_churn_model(str(OPERACIONAL_PATH), str(HISTORICO_PATH))
elif metric_files:
    rows = []
    for metric_file in metric_files:
        for line in metric_file.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 2:
                rows.append(float(parts[1]))
    auc_roc = max(rows)
else:
    _, auc_roc = train_churn.train_churn_model(str(OPERACIONAL_PATH), str(HISTORICO_PATH))

print(f'AUC-ROC churn: {auc_roc:.4f}')
if Path('reports/precision_recall_churn').exists():
    display(Image(filename='reports/precision_recall_churn'))


## 6. MLflow

O MLflow registra experimentos de segmentacao, classificacao de perfil e churn. A celula abaixo resume runs locais existentes e pode registrar todos os experimentos caso o diretorio `mlruns/` ainda nao exista.


In [ ]:
mlflow_tracking = SourceFileLoader('mlflow_tracking', 'src/pipeline/mlflow_tracking.py').load_module()

if not Path('mlruns').exists():
    mlflow_tracking.register_all_experiments()

mlflow.set_tracking_uri('file:./mlruns')
client = mlflow.tracking.MlflowClient()
experiments = [exp for exp in client.search_experiments() if exp.lifecycle_stage == 'active']
rows = []
for exp in experiments:
    runs = client.search_runs([exp.experiment_id], order_by=['start_time DESC'], max_results=20)
    rows.append({'experiment': exp.name, 'experiment_id': exp.experiment_id, 'runs': len(runs)})
mlflow_summary = pd.DataFrame(rows).sort_values('experiment') if rows else pd.DataFrame(columns=['experiment', 'experiment_id', 'runs'])
display(mlflow_summary)


## 7. Conclusoes

A segmentacao entrega os quatro perfis operacionais para o Dock 360: fiel, economico, esquecido e abandono. A classificacao antecipa o perfil no momento da compra, enquanto o modelo de churn prioriza clientes para acao comercial e pos-venda.


In [ ]:
perfil_summary = historico.merge(labels_df, left_on='id_cliente', right_on='cliente_id').groupby('perfil_cluster').agg(
    clientes=('id_cliente', 'count'),
    churn_medio=('churn_rede_24m', 'mean'),
    share_rede=('share_revisoes_rede_24m', 'mean'),
    revisoes_24m=('qtde_revisoes_24m', 'mean'),
    gasto_rede=('gasto_manutencao_rede_24m', 'mean'),
    satisfacao=('satisfacao_marca_24m', 'mean'),
).reset_index()
perfil_summary['pct_clientes'] = perfil_summary['clientes'] / perfil_summary['clientes'].sum()
display(perfil_summary.round(4))

print('Metricas finais para a banca')
print(f'- ARI clustering: {ari:.4f}')
print(f'- F1 Macro classificacao de perfil: {best_f1_macro:.4f}')
print(f'- AUC-ROC churn: {auc_roc:.4f}')
print('- Perfis Dock 360: fiel, economico, esquecido, abandono')
